# Inference Profile Setup Notebook

This notebook creates AWS Bedrock inference profiles programmatically and tests them with a LangGraph agent.

**Features:**
- Easy model selection via commentable constants
- Robust logging for troubleshooting
- Saves ARN to `.env` file for use by other notebooks
- Full LangGraph agent test

## Part 1: Inference Profile Setup

In [ ]:
# =============================================================================
# MODEL SELECTION - Uncomment ONE model to use
# =============================================================================

# MODEL_KEY = "haiku"      # Fast & cheap - good for testing
MODEL_KEY = "sonnet"       # Balanced - recommended for production (DEFAULT)
# MODEL_KEY = "sonnet4"    # Claude Sonnet 4 - latest generation

# =============================================================================
# MODEL ID MAPPING (Cross-region inference profile IDs - DO NOT EDIT)
# =============================================================================
# These are cross-region inference profile model IDs (with us. prefix)
MODEL_IDS = {
    "haiku": "us.anthropic.claude-3-5-haiku-20241022-v1:0",
    "sonnet": "us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    "sonnet4": "us.anthropic.claude-sonnet-4-20250514-v1:0",
}

# =============================================================================
# AWS REGION (change if needed)
# =============================================================================
AWS_REGION = "us-west-2"

# Profile naming prefix
PROFILE_PREFIX = "langgraph-lab"

print(f"Selected model: {MODEL_KEY}")
print(f"Model ID: {MODEL_IDS[MODEL_KEY]}")
print(f"Region: {AWS_REGION}")

In [ ]:
import logging
import json
import boto3
from botocore.exceptions import ClientError, BotoCoreError

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

# Set boto3 logging to WARNING to reduce noise
logging.getLogger('boto3').setLevel(logging.WARNING)
logging.getLogger('botocore').setLevel(logging.WARNING)

logger.info("Logging configured successfully")

In [ ]:
def get_aws_clients():
    """Initialize AWS clients with error handling."""
    logger.info("Initializing AWS clients...")
    try:
        sts = boto3.client('sts', region_name=AWS_REGION)
        bedrock = boto3.client('bedrock', region_name=AWS_REGION)
        bedrock_runtime = boto3.client('bedrock-runtime', region_name=AWS_REGION)

        # Verify credentials
        identity = sts.get_caller_identity()
        account_id = identity['Account']
        logger.info(f"AWS Account: {account_id}")
        logger.info(f"Region: {AWS_REGION}")

        return sts, bedrock, bedrock_runtime, account_id
    except Exception as e:
        logger.error(f"Failed to initialize AWS clients: {e}")
        raise

sts, bedrock, bedrock_runtime, ACCOUNT_ID = get_aws_clients()

In [ ]:
def get_profile_name(model_key: str) -> str:
    """Generate profile name."""
    return f"{PROFILE_PREFIX}-{model_key}"


def get_model_arn(model_id: str) -> str:
    """
    Generate the source model ARN for creating an inference profile.
    
    Uses the cross-region inference profile format:
    arn:aws:bedrock:{region}:{account}:inference-profile/{model_id}
    """
    return f"arn:aws:bedrock:{AWS_REGION}:{ACCOUNT_ID}:inference-profile/{model_id}"


def get_profile_arn(profile_name: str) -> str:
    """Construct the expected application inference profile ARN."""
    return f"arn:aws:bedrock:{AWS_REGION}:{ACCOUNT_ID}:application-inference-profile/{profile_name}"


def get_existing_profile(profile_name: str) -> dict | None:
    """
    Check if profile exists by trying to get it directly.
    
    This approach works even without ListInferenceProfiles permission.
    """
    profile_arn = get_profile_arn(profile_name)
    logger.info(f"Checking for existing profile: {profile_name}")
    
    try:
        response = bedrock.get_inference_profile(inferenceProfileIdentifier=profile_arn)
        logger.info(f"Profile '{profile_name}' already exists")
        return {
            'inferenceProfileName': response['inferenceProfileName'],
            'inferenceProfileArn': response['inferenceProfileArn'],
        }
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'ResourceNotFoundException':
            logger.info(f"Profile '{profile_name}' does not exist")
            return None
        elif error_code == 'AccessDeniedException':
            # Can't check - will try to create and handle conflict
            logger.warning("No permission to get profile, will try to create")
            return None
        else:
            logger.error(f"Error checking profile: {e}")
            raise


def list_existing_profiles() -> dict:
    """
    List all application inference profiles.
    
    Note: This requires bedrock:ListInferenceProfiles permission.
    If permission is denied, returns empty dict.
    """
    logger.info("Listing existing inference profiles...")
    try:
        response = bedrock.list_inference_profiles(typeEquals='APPLICATION')
        profiles = {p['inferenceProfileName']: p for p in response.get('inferenceProfileSummaries', [])}
        logger.info(f"Found {len(profiles)} existing profile(s)")
        return profiles
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'AccessDeniedException':
            logger.warning("No permission to list profiles (this is OK)")
            return {}
        logger.error(f"Failed to list profiles: {e}")
        raise


logger.info("Profile management functions defined")

In [ ]:
def create_inference_profile(model_key: str, force_recreate: bool = False) -> str:
    """
    Create an inference profile for the specified model.

    Args:
        model_key: One of 'haiku', 'sonnet', 'sonnet4', 'sonnet45'
        force_recreate: If True, delete existing profile and create new one

    Returns:
        The inference profile ARN
    """
    if model_key not in MODEL_IDS:
        raise ValueError(f"Invalid model_key: {model_key}. Must be one of {list(MODEL_IDS.keys())}")

    model_id = MODEL_IDS[model_key]
    profile_name = get_profile_name(model_key)
    model_arn = get_model_arn(model_id)

    logger.info("=" * 60)
    logger.info(f"Model: {model_key}")
    logger.info(f"Model ID: {model_id}")
    logger.info(f"Profile Name: {profile_name}")
    logger.info("=" * 60)

    # Check for existing profile
    existing = get_existing_profile(profile_name)
    if existing:
        if force_recreate:
            logger.warning("force_recreate=True, deleting existing profile...")
            delete_inference_profile(model_key)
        else:
            arn = existing['inferenceProfileArn']
            logger.info(f"Using existing profile: {arn}")
            return arn

    # Create new profile
    logger.info("Creating new inference profile...")
    try:
        tags = [
            {'key': 'AmazonBedrockManaged', 'value': 'true'},  # Required for SageMaker Studio
            {'key': 'Purpose', 'value': 'LangGraphLab'},
            {'key': 'Model', 'value': model_key},
        ]

        response = bedrock.create_inference_profile(
            inferenceProfileName=profile_name,
            modelSource={'copyFrom': model_arn},
            description=f"LangGraph Lab profile for {model_key}",
            tags=tags
        )

        arn = response['inferenceProfileArn']
        logger.info("Created profile successfully!")
        logger.info(f"ARN: {arn}")
        return arn

    except ClientError as e:
        error_code = e.response['Error']['Code']
        error_msg = e.response['Error']['Message']
        
        # Handle "already exists" errors (profile exists but we couldn't check earlier)
        if error_code in ('ConflictException', 'ResourceInUseException'):
            logger.info("Profile already exists, retrieving ARN...")
            arn = get_profile_arn(profile_name)
            logger.info(f"Using existing profile: {arn}")
            return arn
        
        logger.error(f"AWS Error [{error_code}]: {error_msg}")
        raise
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        raise


logger.info("create_inference_profile function defined")

In [ ]:
def delete_inference_profile(model_key: str) -> bool:
    """Delete an inference profile."""
    profile_name = get_profile_name(model_key)
    profile_arn = get_profile_arn(profile_name)

    existing = get_existing_profile(profile_name)
    if not existing:
        logger.warning(f"Profile '{profile_name}' not found, nothing to delete")
        return False

    logger.info(f"Deleting profile: {profile_arn}")

    try:
        bedrock.delete_inference_profile(inferenceProfileIdentifier=profile_arn)
        logger.info("Profile deleted successfully")
        return True
    except ClientError as e:
        logger.error(f"Failed to delete profile: {e}")
        raise


logger.info("delete_inference_profile function defined")

In [ ]:
def test_inference_profile(profile_arn: str) -> bool:
    """Test the inference profile with a simple prompt."""
    logger.info("Testing inference profile...")

    try:
        response = bedrock_runtime.converse(
            modelId=profile_arn,
            messages=[
                {
                    'role': 'user',
                    'content': [{'text': 'Say hello in exactly 3 words.'}]
                }
            ],
            inferenceConfig={
                'maxTokens': 50,
                'temperature': 0.0
            }
        )

        output_text = response['output']['message']['content'][0]['text']
        usage = response.get('usage', {})

        logger.info("Test successful!")
        logger.info(f"Response: {output_text}")
        logger.info(f"Input tokens: {usage.get('inputTokens', 'N/A')}")
        logger.info(f"Output tokens: {usage.get('outputTokens', 'N/A')}")
        return True

    except ClientError as e:
        error_code = e.response['Error']['Code']
        error_msg = e.response['Error']['Message']
        logger.error(f"Test failed [{error_code}]: {error_msg}")
        return False
    except Exception as e:
        logger.error(f"Test failed with unexpected error: {e}")
        return False


logger.info("test_inference_profile function defined")

In [ ]:
def save_profile_to_env(profile_arn: str, model_key: str) -> str:
    """Save the profile ARN to a .env file for use by other notebooks."""
    env_file = f".inference-profile-{model_key}.env"

    content = f"""# Auto-generated by setup_inference_profile.ipynb
INFERENCE_PROFILE_ARN="{profile_arn}"
AWS_REGION="{AWS_REGION}"
MODEL_KEY="{model_key}"
"""

    with open(env_file, 'w') as f:
        f.write(content)

    logger.info(f"Saved configuration to {env_file}")
    return env_file


logger.info("save_profile_to_env function defined")

## Create, Test & Save the Inference Profile

Run the cell below to:
1. Create the inference profile (or use existing one)
2. Test it with a simple prompt
3. Save the ARN to a `.env` file

In [ ]:
# Create the inference profile for the selected model
INFERENCE_PROFILE_ARN = create_inference_profile(MODEL_KEY)

# Save to .env file
env_file = save_profile_to_env(INFERENCE_PROFILE_ARN, MODEL_KEY)

# Test the profile
test_inference_profile(INFERENCE_PROFILE_ARN)

# Display for copy-paste
print(f"\n{'='*60}")
print(f'INFERENCE_PROFILE_ARN = "{INFERENCE_PROFILE_ARN}"')
print(f"Saved to: {env_file}")
print(f"{'='*60}\n")

## Utility: List All Lab Profiles

Run the cell below to see all existing `langgraph-lab-*` inference profiles.

In [ ]:
# Utility: List all existing lab profiles
# Note: This requires bedrock:ListInferenceProfiles permission
profiles = list_existing_profiles()

if not profiles:
    print("No profiles found (or no permission to list).")
    print(f"\nTo check a specific profile, you can use:")
    print(f"  get_existing_profile('{get_profile_name(MODEL_KEY)}')")
else:
    lab_profiles = {k: v for k, v in profiles.items() if k.startswith(PROFILE_PREFIX)}
    if lab_profiles:
        print(f"Found {len(lab_profiles)} lab profile(s):")
        for name, profile in lab_profiles.items():
            print(f"  - {name}: {profile['inferenceProfileArn']}")
    else:
        print("No lab profiles found.")

---

## Part 2: LangGraph Agent Test

Now that the inference profile is created, let's test it with a full LangGraph agent that can use tools.

In [ ]:
%pip install langgraph>=1.0.6 langchain-aws -q

In [ ]:
from typing import Literal
from datetime import datetime

from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

logger.info("LangGraph imports successful!")

In [ ]:
@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


tools = [get_current_time, add_numbers]
logger.info(f"Defined {len(tools)} tools: {[t.name for t in tools]}")

In [ ]:
# Initialize LLM with the inference profile
llm = ChatBedrockConverse(
    model=INFERENCE_PROFILE_ARN,
    provider="anthropic",
    region_name=AWS_REGION,
    temperature=0,
)
llm_with_tools = llm.bind_tools(tools)


# Define graph functions
def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    """Determine whether to continue to tools or end."""
    last_message = state["messages"][-1]
    return "tools" if last_message.tool_calls else END


def call_model(state: MessagesState):
    """Call the LLM."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# Build the graph
graph = StateGraph(MessagesState)
graph.add_node("agent", call_model)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue)
graph.add_edge("tools", "agent")

agent = graph.compile()
logger.info("Agent graph compiled successfully!")

In [ ]:
def run_agent(question: str):
    """Run the agent with a question and display the response."""
    logger.info(f"Question: {question}")
    print(f"Question: {question}")
    print("-" * 50)

    result = agent.invoke({
        "messages": [
            SystemMessage(content="You are a helpful assistant. Use tools when needed."),
            HumanMessage(content=question),
        ]
    })

    final_message = result["messages"][-1]
    print(f"\nResponse:\n{final_message.content}")
    return result

## Test the Agent

Run the cells below to test the agent with tool-calling capabilities.

In [ ]:
# Test the full agent with both tools
result = run_agent("What is the current time and what is 42 + 17?")

In [ ]:
# Try your own question here
my_question = "Add 999 and 1, then tell me the time."

result = run_agent(my_question)